# Árvores urbanas — experimento da área completa (29/10/2026)

**Proposta 1:** segmentar a vegetação urbana e distinguir árvores de gramíneas com **Quickshift, K-Means, NDVI, brilho HSV, bordas Canny e Random Forest**.

**Proposta 2:** estimar, experimentalmente, alturas a partir da área da copa e localizar árvores próximas aos **postes**, nas distâncias de 1, 3 e 5 metros.

O conjunto de entrada **Area1** contém imagens RGB e RGB+NIR, amostras pontuais de árvores e gramíneas, área de interesse e postes. **Não contém linhas/cabos de energia**. A proximidade de postes não representa detecção de contato com cabos nem risco elétrico confirmado.

Os resultados anteriormente observados no notebook fornecido (458 pares objeto–rótulo, 92 testes, acurácia 87%) são **históricos, não reexecutados aqui**. Este notebook corrige duplicações de rótulos por objeto e restringe a etapa de altura apenas à classe árvore; por isso, suas novas métricas **podem ser diferentes**.

**Recursos:** execução integral de ~50 milhões de pixels requer uma sessão com bastante RAM, preferencialmente Google Colab com RAM alta. Execute as células na ordem. Os dados pesados permanecem no Google Drive; somente o código está no GitHub.

In [ ]:
# 0 — Instalação, bibliotecas e dados
%pip -q install rasterio geopandas scikit-image scikit-learn matplotlib pandas shapely

from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio import features
from rasterio.mask import mask
from rasterio.windows import Window
from shapely.geometry import shape
from skimage.segmentation import quickshift
from skimage.feature import canny
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Fora do Colab: use um diretório local para DATA_DIR.')

# Ajuste esta variável se tiver feito o atalho da pasta compartilhada no Meu Drive.
DATA_DIR = Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1')
OUT_DIR = DATA_DIR.parent / 'outputs'
ARQUIVOS = ['AOI_RGB_NEW.tif','AOI_NEW_MULTI.tif','AOI_new.shp',
            'AOI_new.shx','AOI_new.dbf','AOI_new.prj',
            'Arvores.gpkg','Grama.gpkg','postes.gpkg']
faltantes = [n for n in ARQUIVOS if not (DATA_DIR/n).is_file()]
if faltantes:
    raise FileNotFoundError(f'Coloque a pasta Area1 no Google Drive e ajuste DATA_DIR. Faltam: {faltantes}')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dados:', DATA_DIR, '| Resultados:', OUT_DIR)

In [ ]:
# 1–3 — Quickshift e polígonos georreferenciados
with rasterio.open(DATA_DIR / 'AOI_RGB_NEW.tif') as rgb:
    im = rgb.read().transpose(1, 2, 0)
    transform_rgb, crs_rgb = rgb.transform, rgb.crs

if crs_rgb is None or not crs_rgb.is_projected:
    raise ValueError('O raster RGB deve possuir CRS projetado em metros.')
AOI = gpd.read_file(DATA_DIR / 'AOI_new.shp').to_crs(crs_rgb)
if AOI.empty:
    raise ValueError('A área de interesse está vazia.')

# Parâmetros mantidos do experimento do pesquisador.
segments_quick2 = quickshift(im, kernel_size=3, max_dist=5, ratio=0.04).astype('int32')
del im
# Rasterio.features.shapes aceita int32 (não aceita uint32).
polygons = (shape(geojson) for geojson, value in
            features.shapes(segments_quick2, transform=transform_rgb))
Poly_gdf = gpd.GeoDataFrame(geometry=list(polygons), crs=crs_rgb)
del segments_quick2
clip_poly = Poly_gdf.clip(AOI).reset_index(drop=True)
del Poly_gdf
clip_poly = clip_poly.loc[~clip_poly.geometry.is_empty].copy()
print('Segmentos na área:', len(clip_poly), 'CRS:', crs_rgb)
if clip_poly.empty:
    raise ValueError('Nenhum segmento intercepta a área de interesse.')

In [ ]:
# 4–6 — Características espectrais e K-Means
COLUNAS_KMEANS = ['R_Mean','G_Mean','B_Mean','N_Mean','R_Std','G_Std','B_Std','N_Std']
with rasterio.open(DATA_DIR / 'AOI_NEW_MULTI.tif') as src_mult:
    if src_mult.count < 4 or src_mult.crs != crs_rgb:
        raise ValueError('Raster multiespectral inválido: são necessárias 4 bandas e CRS compatível.')

    def stats_por_objeto(geometria):
        try:
            bloco, _ = mask(src_mult, [geometria.__geo_interface__], crop=True, nodata=0)
        except ValueError:  # Objeto sem sobreposição com o raster multiespectral.
            return [np.nan] * 8
        medias, desvios = [], []
        for banda in bloco[:4]:
            pixels = banda[banda != 0]
            medias.append(float(pixels.mean()) if pixels.size else np.nan)
            desvios.append(float(pixels.std()) if pixels.size else np.nan)
        return medias + desvios

    registros = [stats_por_objeto(g) for g in clip_poly.geometry]

tabela = pd.DataFrame(registros, index=clip_poly.index, columns=COLUNAS_KMEANS)
validos = tabela.dropna().index
if len(validos) < 6:
    raise ValueError('Poucos segmentos válidos para K-Means com seis clusters.')
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
clip_poly['kmeans_cluster'] = np.nan
clip_poly.loc[validos, 'kmeans_cluster'] = kmeans.fit_predict(tabela.loc[validos, COLUNAS_KMEANS])
print('Segmentos por cluster:', clip_poly.kmeans_cluster.value_counts().to_dict())

# Cluster 3 reproduz a opção experimental; o número do cluster não é fixo entre conjuntos.
CLUSTER_VEGETACAO = 3
candidatos = clip_poly.loc[clip_poly.kmeans_cluster == CLUSTER_VEGETACAO].copy()
if candidatos.empty:
    raise ValueError('Cluster 3 sem candidatos; revise visualmente os clusters de vegetação.')
print('Candidatos à vegetação:', len(candidatos))

In [ ]:
# 7–10 — Rótulos e composição das características (processamento em blocos)
arvores = gpd.read_file(DATA_DIR / 'Arvores.gpkg').to_crs(crs_rgb)
grama = gpd.read_file(DATA_DIR / 'Grama.gpkg').to_crs(crs_rgb)
arvores['Classe'], grama['Classe'] = 'arvore', 'grama'
rotulos = gpd.GeoDataFrame(pd.concat(
    [arvores[['Classe','geometry']], grama[['Classe','geometry']]],
    ignore_index=True), crs=crs_rgb)
pares = gpd.sjoin(candidatos, rotulos, how='inner', predicate='intersects')
print('Pares objeto–rótulo:', len(pares), '| Objetos únicos:', pares.index.nunique())

# Um mesmo objeto pode conter diversos pontos rotulados; removemos repetições e
# ambiguidades para não contaminar treino/teste com o mesmo segmento.
classes_por_objeto = pares.groupby(level=0)['Classe'].agg(lambda x: sorted(set(x)))
ids_confiaveis = classes_por_objeto[classes_por_objeto.map(len) == 1].index
amostras = candidatos.loc[ids_confiaveis].copy()
amostras['Classe'] = classes_por_objeto.loc[ids_confiaveis].map(lambda x: x[0])
print('Objetos rotulados sem ambiguidade:', len(amostras),
      '| Classes:', amostras.Classe.value_counts().to_dict())
if amostras.Classe.nunique() < 2:
    raise ValueError('É necessário haver amostras de árvores e gramíneas no cluster escolhido.')

# O raster é grande: calcular NDVI/brilho/Canny em blocos para evitar
# carregar bandas e derivados em ponto flutuante de toda a área ao mesmo tempo.
features_path = OUT_DIR / 'features_stack.tif'
TAMANHO_BLOCO, HALO = 1024, 8
with rasterio.open(DATA_DIR / 'AOI_NEW_MULTI.tif') as src:
    if src.count < 4 or src.crs != crs_rgb:
        raise ValueError('A imagem multiespectral deve ter 4 bandas e CRS projetado.')
    profile = src.profile.copy()
    profile.update(count=3, dtype='float32', nodata=None, compress='deflate')
    # No arquivo fonte uint16, rgb2hsv(...)[...,2] é max(R,G,B)/65535.
    valor_maximo_rgb = float(np.iinfo(np.dtype(src.dtypes[0])).max)
    with rasterio.open(features_path, 'w', **profile) as dst:
        for y in range(0, src.height, TAMANHO_BLOCO):
            for x in range(0, src.width, TAMANHO_BLOCO):
                h = min(TAMANHO_BLOCO, src.height - y)
                w = min(TAMANHO_BLOCO, src.width - x)
                x0, y0 = max(0, x-HALO), max(0, y-HALO)
                x1, y1 = min(src.width, x+w+HALO), min(src.height, y+h+HALO)
                bloco = src.read(window=Window(x0,y0,x1-x0,y1-y0))
                r = bloco[0].astype('float32')
                nir = bloco[3].astype('float32')
                ndvi = (nir-r) / (nir+r+1e-8)
                brilho = np.max(bloco[:3], axis=0).astype('float32') / valor_maximo_rgb
                bordas = canny(nir / 10000.0, sigma=1)
                ox, oy = x-x0, y-y0
                dados = np.stack([
                    brilho[oy:oy+h, ox:ox+w],
                    bordas[oy:oy+h, ox:ox+w].astype('float32'),
                    ndvi[oy:oy+h, ox:ox+w]
                ]).astype('float32')
                dst.write(dados, window=Window(x,y,w,h))
print('Pilhas de características gravadas em:', features_path)
print('Aviso: Canny em blocos usa halo de 8 pixels; efeitos de borda podem diferir ligeiramente do cálculo integral.')

In [ ]:
# 11–12 — Random Forest, matriz de confusão e predição por objeto
def extrair_features(gdf, raster_path):
    resultados = []
    with rasterio.open(raster_path) as src:
        for geom in gdf.geometry:
            try:
                bloco, _ = mask(src, [geom.__geo_interface__], crop=True, nodata=-999)
                medias = []
                for banda in bloco:
                    px = banda[banda != -999]
                    medias.append(float(px.mean()) if px.size else np.nan)
            except ValueError:
                medias = [np.nan] * 3
            resultados.append(medias)
    return pd.DataFrame(resultados, index=gdf.index,
                        columns=['mean_intensity','mean_edges','mean_ndvi'])

atributos = extrair_features(amostras, features_path)
atributos['classe'] = amostras['Classe'].to_numpy()
atributos = atributos.dropna()
X, y = atributos.drop(columns='classe'), atributos['classe']
if y.nunique() < 2 or y.value_counts().min() < 2:
    raise ValueError('Amostras insuficientes em uma das classes para separar treino/teste.')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
rf_classifier = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train, y_train)
pred_teste = rf_classifier.predict(X_test)
print(f'Objetos distintos: {len(X)} | Treino: {len(X_train)} | Teste: {len(X_test)}')
print(classification_report(y_test, pred_teste, zero_division=0))
print('Acurácia:', round(accuracy_score(y_test, pred_teste), 4))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_teste, ax=ax, cmap='Blues')
fig.savefig(OUT_DIR / 'matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()
metricas = classification_report(y_test, pred_teste, zero_division=0, output_dict=True)
(OUT_DIR / 'metricas_classificacao.json').write_text(
    json.dumps({'n_treino':len(X_train),'n_teste':len(X_test),
                'relatorio':metricas}, ensure_ascii=False, indent=2),
    encoding='utf-8')

atrib_pred = extrair_features(candidatos, features_path).dropna()
candidatos['predicted_classe'] = pd.Series(index=candidatos.index, dtype='object')
if not atrib_pred.empty:
    candidatos.loc[atrib_pred.index, 'predicted_classe'] = rf_classifier.predict(atrib_pred)
print('Previsões:', candidatos.predicted_classe.value_counts(dropna=False).to_dict())

In [ ]:
# 13–15 — Área da copa e altura empírica (configuração C3)
# Somente objetos classificados como árvore seguem para esta etapa.
tree_polys = candidatos.loc[candidatos.predicted_classe == 'arvore'].copy()
tree_polys['area_m2'] = tree_polys.geometry.area
tree_polys['altura_estimada_m'] = 2.5 * tree_polys['area_m2'] ** 0.4
filter_tree = tree_polys.loc[tree_polys.altura_estimada_m > 7].copy()
print('Árvores previstas:', len(tree_polys),
      '| Altura estimada >7m:', len(filter_tree))
if not filter_tree.empty:
    filter_tree.to_file(OUT_DIR/'arvores_estimadas_maiores_7m.gpkg', driver='GPKG')
else:
    print('Nenhuma árvore passou no filtro; arquivo GPKG não gerado.')
# Atenção: estimativa alométrica NÃO validada em campo.

In [ ]:
# 16–17 — Análise de proximidade a POSTES (não aos cabos)
postes = gpd.read_file(DATA_DIR/'postes.gpkg').to_crs(crs_rgb).clip(AOI)
print('Postes na área:', len(postes), '| Árvores selecionadas:', len(filter_tree))
resumo = []
for distancia in (1, 3, 5):
    if filter_tree.empty or postes.empty:
        vizinhos = gpd.GeoDataFrame(columns=['geometry'], geometry='geometry', crs=crs_rgb)
    else:
        vizinhos = gpd.sjoin_nearest(filter_tree, postes, how='inner',
                                     max_distance=distancia, distance_col='distancia_m')
    qtd = vizinhos.index.nunique()
    resumo.append({'distancia_m':distancia,'arvores_unicas':int(qtd),
                   'pares_arvore_poste':len(vizinhos)})
    print(f'Até {distancia}m: {qtd} árvores únicas; {len(vizinhos)} pares árvore–poste')
    if not vizinhos.empty:
        vizinhos.to_file(OUT_DIR / f'arvores_proximas_postes_{distancia}m.gpkg', driver='GPKG')
    else:
        print(f'Sem pares a {distancia}m; nenhum GPKG vazio será criado.')
pd.DataFrame(resumo).to_csv(OUT_DIR/'resumo_proximidade_postes.csv', index=False)
print('Concluído. Arquivos em', OUT_DIR)

## Interpretação e limites da avaliação

- O conjunto de **458 pares rotulados e acurácia de 87%** pertence à execução original recebida em outubro, e não deve ser confundido com as **novas métricas** geradas após a deduplicação e a divisão estratificada.
- A separação aleatória por objeto reduz duplicações, mas **não substitui validação espacial independente**. Regiões vizinhas podem apresentar autocorrelação.
- O cluster 3 é uma escolha empírica sujeita a inspeção. Rótulos numéricos de K-Means não têm significado universal.
- A equação de altura `2.5 * area**0.4` é uma **adaptação experimental ainda não calibrada** por alturas observadas.
- Distâncias a **postes** são medidas em planta (2D), e não confirmam contato/interferência com **condutores elétricos**.
- Os resultados e os arquivos produzidos ficam em `outputs/`, no Drive, **não** no GitHub.
- Para próximas avaliações: comparar SVM/outros classificadores, realizar ablação das três variáveis, introduzir divisão espacial de treino/teste e validar a estimativa de altura com medições de campo.

Consulte `README.md`, `data/README.md` e `docs/allometria.md`.